In [ ]:
# --- MAIN PIPELINE CONTROLLER ---

# 1. IMPORTS
import step1_loading
import step2_preprocessing
import step3_epoching
import step4_pseudotrials
import step5_decoding
import step6_visualization
import step7_comparison
import matplotlib.pyplot as plt
import numpy as np

# 2. CONFIGURATION
SUBJECT_ID = 'sub-17'
BASIC_PATH = 'project/ds006761'

print(f"--- STARTING DUAL-PLAYER ANALYSIS: {SUBJECT_ID} ---")

# =================================================================
# STEP 1: LOAD FULL DATASET
# =================================================================
# Load the 143-channel file and split into two EEG objects using the new layout
raw_p1_full, raw_p2_full = step1_loading.load_and_split_data(SUBJECT_ID, BASIC_PATH)

# Store results for final comparison: { player_num: (times, scores) }
final_scores = {}

# =================================================================
# LOOP: PROCESS PLAYER 1 THEN PLAYER 2
# =================================================================
for player_num, raw_data in zip([1, 2], [raw_p1_full, raw_p2_full]):
    
    print(f"\n" + "="*50)
    print(f"   PROCESSING PLAYER {player_num}")
    print("="*50)

    if raw_data is None:
        print(f"❌ Missing data for Player {player_num}. Skipping.")
        continue

    # A. VISUALIZE RAW
    step1_loading.visualize_raw_data(raw_data, f"Player {player_num} (Raw)")

    # B. PREPROCESSING (Paper Replication)
    # No Filter, Interpolate, CAR, Resample
    raw_clean = step2_preprocessing.run_preprocessing(raw_data)
    step2_preprocessing.visualize_clean_data(raw_clean, f"Player {player_num} (Cleaned)")
    step2_preprocessing.visualize_ica_components(raw_clean, 15)

    # C. EPOCHING (Now returns a TUPLE)
    # We store the tuple in 'epochs_tuple'
    epochs_tuple, full_events_df = step3_epoching.run_epoching(raw_clean, SUBJECT_ID, BASIC_PATH, player_num)

    # ... inside the main loop ...

    # D. PSEUDO-TRIALS
    epochs_binned = step4_pseudotrials.create_pseudo_trials(epochs_tuple)

    # --- THE FIX IS HERE ---
    # 1. Extract Labels from the DataFrame
    # We need the 'playerX_resp' column that matches the valid trials
    current_labels = full_events_df[f'player{player_num}_resp'].values

    # 2. Pass these labels to Step 5
    times, scores = step5_decoding.run_svm_decoding(epochs_binned, custom_labels=current_labels)

    final_scores[player_num] = (times, scores)
    # -----------------------

    # F. VISUALIZATION
    step6_visualization.plot_paper_replication(times, scores, f"Player {player_num}")


    # G. COMPARATIVE ANALYSIS (Now Fixed)
    # We pass 'epochs_binned' (Array) and 'full_events_df' (Table)
    comp_results = step7_comparison.run_comparative_analysis(epochs_binned, full_events_df, player_num)
    
    # This will now plot the 4 lines with the correct 0-5s axis
    step7_comparison.plot_comparisons(comp_results, player_num)

# =================================================================
# FINAL STEP: HEAD-TO-HEAD VISUALIZATION
# =================================================================
print("\n" + "="*50)
print("   GENERATING FINAL COMPARISON: P1 vs P2")
print("="*50)

if 1 in final_scores and 2 in final_scores:
    t1, s1 = final_scores[1]
    t2, s2 = final_scores[2]
    
    plt.figure(figsize=(12, 6))
    
    # Plot Player 1
    plt.plot(t1, s1, label='Player 1 Accuracy', color='blue', linewidth=2)
    
    # Plot Player 2
    plt.plot(t2, s2, label='Player 2 Accuracy', color='orange', linewidth=2)
    
    # Reference Lines
    plt.axhline(33.33, color='k', linestyle='--', label='Chance (33%)') # 1/3 Chance
    plt.axvline(0, color='r', linestyle='-', alpha=0.5, label='Decision Onset')
    
    # Styling
    plt.title(f"Head-to-Head Decoding Performance: {SUBJECT_ID}")
    plt.xlabel("Time (s)")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.show()
    
    # Print Summary Stats
    p1_max = np.max(s1)
    p2_max = np.max(s2)
    print(f"Player 1 Max Accuracy: {p1_max:.2f}%")
    print(f"Player 2 Max Accuracy: {p2_max:.2f}%")
    
    if p1_max > p2_max:
        print("🏆 Player 1's brain was more readable (higher decoding accuracy).")
    else:
        print("🏆 Player 2's brain was more readable (higher decoding accuracy).")

else:
    print("Could not compare players (one or both missing).")

print("\n✅ Full Dual-Player Pipeline Complete.")

In [ ]:
# =================================================================
# 1. CONFIGURATION (MULTI-SUBJECT DYNAMIC)
# =================================================================
import step1_loading
import step2_preprocessing
import step3_epoching
import step4_pseudotrials
import step5_decoding
import step6_visualization
import step7_comparison
import numpy as np

# Dynamically generate Subject IDs from 01 to 34
START_SUB = 1
END_SUB = 2
SUBJECT_IDS = [f"sub-{i:02d}" for i in range(START_SUB, END_SUB + 1)]

BASIC_PATH = 'project/ds006761'

print(f"--- CONFIGURATION ---")
print(f"Target Subjects: {len(SUBJECT_IDS)} subjects")
print(f"Range: {SUBJECT_IDS[0]} to {SUBJECT_IDS[-1]}")

# Storage for Grand Average Analysis
# 1. Main Decoding
group_results = {1: [], 2: []} 

# 2. Comparative Decoding (New)
# Structure: { Player: { 'Own Current': [], ... } }
keys = ['Own Current', 'Opponent Current', 'Own Previous', 'Opponent Previous']
group_comp_results = {
    1: {k: [] for k in keys},
    2: {k: [] for k in keys}
}

print(f"\n--- STARTING PIPELINE ---")

# =================================================================
# 2. OUTER LOOP: SUBJECTS
# =================================================================
for subject_id in SUBJECT_IDS:
    print(f"\n" + "#"*60)
    print(f"   PROCESSING SUBJECT: {subject_id}")
    print("#"*60)
    
    # Load Data for this Subject
    try:
        raw_p1_full, raw_p2_full = step1_loading.load_and_split_data(subject_id, BASIC_PATH)
    except Exception as e:
        print(f"❌ Failed to load {subject_id}: {e}")
        continue

    # =============================================================
    # 3. INNER LOOP: PLAYERS (1 & 2)
    # =============================================================
    for player_num, raw_data in zip([1, 2], [raw_p1_full, raw_p2_full]):
        
        if raw_data is None:
            print(f"   -> Skipping Player {player_num} (No Data)")
            continue
            
        print(f"\n   --- Player {player_num} Analysis ---")

        # A. PREPROCESSING + ICA
        raw_clean = step2_preprocessing.run_preprocessing(raw_data)

        # B. EPOCHING
        epochs_tuple, full_df = step3_epoching.run_epoching(raw_clean, subject_id, BASIC_PATH, player_num)

        # C. PSEUDO-TRIALS (Binning)
        epochs_binned = step4_pseudotrials.create_pseudo_trials(epochs_tuple)

        # D. MAIN DECODING (Own Current)
        current_labels = full_df[f'player{player_num}_resp'].values
        times, scores = step5_decoding.run_svm_decoding(epochs_binned, custom_labels=current_labels)
        
        if scores is not None:
            group_results[player_num].append(scores)
            print(f"   -> ✅ Main Scores stored.")
        else:
            print("   -> ❌ Main Decoding failed.")

        # --- G. COMPARATIVE ANALYSIS (Own/Opp, Curr/Prev) ---
        # Run the 4-task analysis for this subject
        comp_results = step7_comparison.run_comparative_analysis(epochs_binned, full_df, player_num)
        
        # Store these scores for the Grand Average
        for task_name, task_scores in comp_results.items():
            if task_scores is not None:
                group_comp_results[player_num][task_name].append(task_scores)
        print("   -> ✅ Comparative Scores stored.")

# =================================================================
# 4. FINAL STEP: GRAND AVERAGE VISUALIZATION
# =================================================================
print("\n" + "="*60)
print("   GENERATING GRAND AVERAGE PLOTS (All Subjects)")
print("="*60)

grand_time_axis = np.linspace(0, 5.0, 20) 

for player_num in [1, 2]:
    print(f"\n--- Player {player_num} Grand Averages ---")
    
    # 1. Plot Main Decoding (Single Line)
    n_subs = len(group_results[player_num])
    if n_subs > 0:
        step6_visualization.plot_grand_average(
            grand_time_axis, 
            group_results[player_num], 
            player_num
        )
    else:
        print(f"No valid main data for Player {player_num}.")

    # 2. Plot Comparative Decoding (4 Lines)
    # This uses the NEW function we added to step7_comparison.py
    step7_comparison.plot_grand_average_comparison(
        group_comp_results[player_num], 
        player_num
    )

print("\n✅ Multi-Subject Analysis Complete.")